## Imports

In [2]:
import numpy as np
import scipy
import sklearn
import pandas as pd
import matplotlib as mpl
from matplotlib import pyplot as pyplot
import json
import os
import copy
from numpy import random as rng

from misc import move

In [3]:
rng = np.random.default_rng()

## Load the data to visualize it

In [4]:
all_data_matrix = pd.read_csv('../Data/raw_data/all_subjects.csv')
subject_1 = all_data_matrix[all_data_matrix['subject']=='A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB']

In [5]:
subject_1_prb10206_7 = subject_1[subject_1['instance']=='prb10206_7']

In [6]:
subject_1_prb10206_7

,subject,event,move,instance,t,piece,target
5180,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,start,0,prb10206_7,1516740048926,-1,-1
5181,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,drag_start,0,prb10206_7,1516740050504,4,2
5182,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,drag_end,0,prb10206_7,1516740050654,4,3
5183,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,drag_start,1,prb10206_7,1516740051006,3,8
5184,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,drag_end,1,prb10206_7,1516740051148,3,2
...,...,...,...,...,...,...,...
17231,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,drag_end,10,prb10206_7,1516740057922,8,16
17232,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,drag_start,11,prb10206_7,1516740058493,8,16
17233,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,win,11,prb10206_7,1516740058526,8,16
17234,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,drag_end,11,prb10206_7,1516740058635,8,16


In [7]:
s1_prb10206_7 = subject_1_prb10206_7.drop('t',axis=1)
s1_prb10206_7.loc[5180]

subject     A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB
event                                               start
move                                                    0
instance                                       prb10206_7
piece                                                  -1
target                                                 -1
Name: 5180, dtype: object

In [8]:
def parse_data_matrix(df):
    df = copy.deepcopy(df)
    df['sq_change'] = df.shift(-1)['target'] - df['target']
    df['dist'] = (df['sq_change'])*((df['sq_change'] % 6) != 0) + (df['sq_change'] // 6)*((df['sq_change'] % 6) == 0)
    df['piece'] = df['piece'].where(df['piece'] != 8, -1)
    df['piece'] = df['piece'].where(df['piece'] == -1, df['piece'] + 1)
    df.loc[df['event'].isin(pd.Series(['win','start','restart'])), ['piece','target','sq_change','dist']] = None
    df['piece'] = df['piece'].astype('Int64')
    df['p_as_str'] = df['piece'].astype('str')
    df['move_obj'] = None
    df['move_obj'] = df['move_obj'].where(df['event']!= 'drag_start',move(df['p_as_str'],df['dist']))
    df['p_as_str'] = df['p_as_str'].where(df['p_as_str'] != '-1', 'r')
    df['move_disp'] = None
    df['move_disp'] = df['move_disp'].where(df['event']!= 'drag_start',df['p_as_str']+df['dist'].astype('Int64').astype('str'))
    df = df[df['event'] != 'drag_end']
    df = df[~((df['event'] == 'win') & (df['event'].shift() == 'win'))]
    df = df[df['dist'] != 0]
    df.loc[df['event'] == 'drag_start','event'] = 'move'
    df = df.drop(columns=['target','sq_change','dist','p_as_str'])
    return df

In [13]:
s1_prb10206_7_processed = parse_data_matrix(s1_prb10206_7)
s1_prb10206_7_processed.to_csv('../Data/my_processed_data/s1_prb10206_7_processed.csv')

In [14]:
processed_data = parse_data_matrix(all_data_matrix.drop('t',axis=1))

In [15]:
processed_data_head = processed_data.head(100)
processed_data_head

,subject,event,move,instance,piece,move_obj,move_disp
0,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,start,0,prb55384_14,<NA>,None,None
1,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,move,0,prb55384_14,3,<misc.move object at 0x000001B68BDD3E10>,31
5,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,move,1,prb55384_14,3,<misc.move object at 0x000001B68BDD3E10>,3-2
15,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,move,2,prb55384_14,3,<misc.move object at 0x000001B68BDD3E10>,32
21,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,move,3,prb55384_14,-1,<misc.move object at 0x000001B68BDD3E10>,r1
...,...,...,...,...,...,...,...
266,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,move,40,prb48146_16,2,<misc.move object at 0x000001B68BDD3E10>,21
268,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,move,41,prb48146_16,5,<misc.move object at 0x000001B68BDD3E10>,5-2
270,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,move,42,prb48146_16,1,<misc.move object at 0x000001B68BDD3E10>,1-3
272,A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB,move,43,prb48146_16,6,<misc.move object at 0x000001B68BDD3E10>,61


In [16]:
processed_data.to_csv('../Data/my_processed_data/processed_data.csv')